# Pólya Urn Model of the FTTL Feedback Loop

**Paper**: Ensign, Friedler, Neville, Scheidegger & Venkatasubramanian (FAccT 2018) *Runaway Feedback Loops in Predictive Policing* (p12)

**Core framework**: The repeated ML process (patrol → discover → retrain) maps to a **generalised Pólya urn**.
The urn represents the system's accumulated belief about which region/segment has higher crime.

| Paper concept | FTTL equivalent |
|---|---|
| Region A / B | `damage_severity` segment (severe / moderate / minor) |
| Police patrol | `model_v1_decision = 1` (scrap decision) |
| Discovered incident | scrapped claim that is a genuine total loss (`pre_ml_label = 1`) |
| Crime rate λ_A | True total loss rate per segment (from `pre_ml_label`) |
| Reported incident | Claims from pre-ML human-decision era (unbiased) |
| Urn balls | Cumulative scrap counts per segment |

**True total loss rates** (from `pre_ml_label`, pre-model ground truth):
- λ_severe ≈ 0.67, λ_moderate ≈ 0.29, λ_minor ≈ 0.21
- Ideal scrap allocation: severe gets λ_severe/(λ_severe+λ_moderate) ≈ **69.8%** of scraps

**Experiments:**
| ID | Method | p12 reference |
|---|---|---|
| E1 | Standard Pólya Urn — Beta convergence | Lemma 2 |
| E2 | Discovered-only urn with non-uniform λ | Lemma 4 |
| E3 | Mixed reported + discovered urn | Eq. 2 & 3, Section 3.4 |
| E4 | FTTL empirical urn trajectory | Fig. 1 analog |
| E5 | Rejection sampling fix | Section 3.5 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

DATA_PATH = '../src/data/synthetic/csv/claims_all.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['claim_date'])

PRE_ML_CUTOFF = pd.Timestamp('2021-06-01')
pre_ml = df[df['claim_date'] < PRE_ML_CUTOFF].copy()

# True total loss rates (λ) per segment — from pre-ML unbiased labels
lambda_seg = pre_ml.groupby('damage_severity')['pre_ml_label'].mean().to_dict()

# --- Convention (matches paper): A = WEAK segment, B = STRONG segment ---
# x = fraction of A-colored (weak/moderate) balls.  Lemma 4: x* → 0.
lam_weak   = lambda_seg['moderate']   # λ_A ≈ 0.283
lam_strong = lambda_seg['severe']     # λ_B ≈ 0.657

# Ideal fraction of moderate in scraps: λ_weak / (λ_weak + λ_strong)
x_ideal = lam_weak / (lam_weak + lam_strong)   # ≈ 0.301

# Small initial counts for theory demonstrations (scaled ratio from pre-ML era)
# Pre-ML era: severe scraps >> moderate scraps, ratio ≈ 5:3
N_STRONG_SMALL, N_WEAK_SMALL = 15, 10   # 10+15=25 balls → Beta converges clearly

# Full-scale initial counts for empirical sections (actual FTTL data)
init_balls = pre_ml.groupby('damage_severity')['model_v1_decision'].sum().astype(int).to_dict()
N_WEAK_FULL   = init_balls['moderate']   # ≈ 1587
N_STRONG_FULL = init_balls['severe']     # ≈ 2667

print(f'λ_weak  (moderate): {lam_weak:.4f}')
print(f'λ_strong (severe):  {lam_strong:.4f}')
print(f'Ideal x* (moderate share): {x_ideal:.4f}  ({x_ideal*100:.1f}%)')
print()
print(f'Demo urn (small): moderate={N_WEAK_SMALL}, severe={N_STRONG_SMALL}')
print(f'Full urn (FTTL):  moderate={N_WEAK_FULL},  severe={N_STRONG_FULL}')
print()
# Pre-ML prior fraction of moderate in scraps
prior_full = N_WEAK_FULL / (N_WEAK_FULL + N_STRONG_FULL)
prior_small = N_WEAK_SMALL / (N_WEAK_SMALL + N_STRONG_SMALL)
print(f'Prior x (moderate fraction): demo={prior_small:.3f}, FTTL={prior_full:.3f}, ideal={x_ideal:.3f}')

---
## E1 — Standard Pólya Urn: Beta(n_A, n_B) Convergence (Lemma 2)

**Lemma 2 (Renlund 2010)**: Starting with $n_A$ weak and $n_B$ strong balls,  
the limiting fraction of weak (A = moderate) balls ~ **Beta($n_A$, $n_B$)**.

**Significance**: The long-run belief depends *only* on initial counts — **initial bias is locked in permanently**.  
Even if the true total loss rates were identical, the system cannot learn this without visiting both segments.

We use small initial counts (n_A=10, n_B=15, T=3000) to show clear Beta convergence.  
Then we show how the FTTL prior (n_A=1587, n_B=2667) locks in a fraction **above** the ideal x* = 0.301.

In [ ]:
def simulate_standard_polya(n_A, n_B, T, n_runs, rng):
    """Standard Pólya urn: draw a ball, return it, add one of same colour. Returns final x=A-fraction."""
    finals = np.empty(n_runs)
    for run in range(n_runs):
        nA, nB = n_A, n_B
        for _ in range(T):
            if rng.random() < nA / (nA + nB):
                nA += 1
            else:
                nB += 1
        finals[run] = nA / (nA + nB)
    return finals

T_E1, N_RUNS = 3000, 2000

# Demo: small initial counts
finals_small = simulate_standard_polya(N_WEAK_SMALL, N_STRONG_SMALL, T_E1, N_RUNS, rng)

# FTTL: full-scale initial counts — Beta is extremely tight
finals_full = simulate_standard_polya(N_WEAK_FULL, N_STRONG_FULL, T_E1, N_RUNS, rng)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: small-count demo — clear Beta convergence
ax = axes[0]
x_grid = np.linspace(0, 1, 500)
beta_small = stats.beta.pdf(x_grid, N_WEAK_SMALL, N_STRONG_SMALL)
ax.hist(finals_small, bins=50, density=True, color='#2196F3', alpha=0.7,
        label=f'simulated ({N_RUNS} runs, T={T_E1})')
ax.plot(x_grid, beta_small, 'r-', lw=2,
        label=f'Beta({N_WEAK_SMALL}, {N_STRONG_SMALL}) — Lemma 2')
ax.axvline(x_ideal, color='green', ls='--', lw=1.5, label=f'ideal x*={x_ideal:.3f}')
ax.axvline(N_WEAK_SMALL/(N_WEAK_SMALL+N_STRONG_SMALL), color='gray', ls=':', lw=1.5,
           label=f'prior={N_WEAK_SMALL/(N_WEAK_SMALL+N_STRONG_SMALL):.3f}')
ks_stat, ks_p = stats.kstest(finals_small, lambda v: stats.beta.cdf(v, N_WEAK_SMALL, N_STRONG_SMALL))
ax.text(0.02, 0.96, f'KS vs Beta:\nstat={ks_stat:.4f}, p={ks_p:.3f}',
        transform=ax.transAxes, va='top', fontsize=8,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.set_xlabel('x = fraction of moderate (weak) balls')
ax.set_ylabel('density')
ax.set_title(f'E1a: Beta({N_WEAK_SMALL},{N_STRONG_SMALL}) convergence (demo)\n'
             f'Prior ({N_WEAK_SMALL/(N_WEAK_SMALL+N_STRONG_SMALL):.3f}) ≠ ideal ({x_ideal:.3f}): bias locked in')
ax.legend(fontsize=8)

# Right: FTTL full-scale — Beta is very tight, prior far from ideal
ax2 = axes[1]
# Beta(1587,2667) is too tight to show as PDF — show as histogram only
prior_full = N_WEAK_FULL / (N_WEAK_FULL + N_STRONG_FULL)
ax2.hist(finals_full, bins=60, density=True, color='#FF9800', alpha=0.8,
         label=f'FTTL (n_A={N_WEAK_FULL}, n_B={N_STRONG_FULL})')
ax2.axvline(x_ideal, color='green', ls='--', lw=2, label=f'ideal x*={x_ideal:.3f}')
ax2.axvline(prior_full, color='gray', ls=':', lw=2, label=f'prior={prior_full:.3f}')
ax2.set_xlabel('x = fraction of moderate balls')
ax2.set_ylabel('density')
ax2.set_title(f'E1b: FTTL full-scale initial prior\n'
              f'Beta({N_WEAK_FULL},{N_STRONG_FULL}) ≈ spike at {prior_full:.3f}, ideal={x_ideal:.3f}\n'
              f'Gap = {prior_full - x_ideal:.3f} → excess moderate share from pre-ML era')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('p12_e1_standard_polya.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'E1a (demo):  KS stat={ks_stat:.4f}, p={ks_p:.3f} — Lemma 2 {"confirmed" if ks_p>0.05 else "borderline"}')
print(f'E1b (FTTL):  prior={prior_full:.4f}, ideal={x_ideal:.4f}, gap={prior_full-x_ideal:+.4f}')

---
## E2 — Discovered-Only Urn with Non-Uniform λ (Lemma 4)

**Lemma 4**: With addition matrix $\begin{pmatrix} \lambda_A & 0 \\ 0 & \lambda_B \end{pmatrix}$ (discovered incidents only),  
the asymptotic fraction x of **weak** (A = moderate) balls → **0** since $\lambda_A < \lambda_B$.

**FTTL**: Only scrapped claims generate total-loss labels.  
- Each step: visit a segment proportional to current ball counts  
- Add a ball only if the visited claim is a genuine total loss (prob = λ_segment)  
- Since λ_severe > λ_moderate, severe balls accumulate faster → x → 0

**Note on convergence speed**: Full collapse (x < 0.05) requires many more steps for small λ gaps.  
We show mean drift from prior (0.5) toward 0 as the Lemma 4 signal — the smaller the gap, the slower the drift.

In [ ]:
def simulate_discovered_only(n_A, n_B, lam_A, lam_B, T, n_runs, rng):
    """
    Discovered-only urn (Lemma 4 / Assumption 3.4).
    A = weak segment (moderate), B = strong segment (severe).
    Returns final x = fraction of A (weak/moderate) balls.
    """
    finals = np.empty(n_runs)
    for run in range(n_runs):
        nA, nB = float(n_A), float(n_B)
        for _ in range(T):
            if rng.random() < nA / (nA + nB):
                if rng.random() < lam_A: nA += 1
            else:
                if rng.random() < lam_B: nB += 1
        finals[run] = nA / (nA + nB)
    return finals

N_RUNS_E2 = 800
prior = N_WEAK_SMALL / (N_WEAK_SMALL + N_STRONG_SMALL)  # 0.40

# Three scenarios at increasing T to show convergence direction
scenarios = [
    (lam_weak, lam_strong, N_WEAK_SMALL, N_STRONG_SMALL,
     f'E2a: moderate(λ={lam_weak:.2f}) vs severe(λ={lam_strong:.2f})\nLarge gap — rapid collapse'),
    (0.20, 0.30, 10, 10,
     'E2b: λ_A=0.20 vs λ_B=0.30\nMedium gap — slower drift toward 0'),
    (0.30, 0.30, 10, 10,
     'E2c: λ_A=λ_B=0.30\nNo gap — no drift (baseline)'),
]

T_vals = [5000, 5000, 5000]
all_finals = [simulate_discovered_only(nA, nB, la, lb, T, N_RUNS_E2, rng)
              for (la, lb, nA, nB, _), T in zip(scenarios, T_vals)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, finals, (la, lb, nA, nB, title), T in zip(axes, all_finals, scenarios, T_vals):
    prior_here = nA / (nA + nB)
    ideal_here = la / (la + lb)
    ax.hist(finals, bins=40, density=True, color='#F44336', alpha=0.75)
    ax.axvline(0.0,        color='black', ls='--', lw=1.8, label='Lemma 4 → 0')
    ax.axvline(ideal_here, color='green', ls='--', lw=1.5, label=f'ideal={ideal_here:.3f}')
    ax.axvline(prior_here, color='gray',  ls=':',  lw=1.5, label=f'prior={prior_here:.3f}')
    drift = prior_here - finals.mean()  # positive = drifted toward 0 ✓
    ax.text(0.04, 0.96,
            f'mean={finals.mean():.3f}\ndrift from prior: {drift:+.3f}',
            transform=ax.transAxes, va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax.set_xlabel('x = fraction of weak (moderate) balls')
    ax.set_ylabel('density')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle(f'E2: Discovered-Only Urn (T={T_vals[0]}, {N_RUNS_E2} runs)\n'
             'Lemma 4: any positive λ gap drives x toward 0 (gap size controls speed)', fontsize=10)
plt.tight_layout()
plt.savefig('p12_e2_discovered_only.png', dpi=150, bbox_inches='tight')
plt.show()

for (la, lb, nA, nB, _), finals, T in zip(scenarios, all_finals, T_vals):
    prior_here = nA/(nA+nB)
    print(f'λ=({la:.2f},{lb:.2f}) prior={prior_here:.3f} → mean={finals.mean():.4f} '
          f'(drift={prior_here-finals.mean():+.4f}, {(finals<0.05).mean()*100:.0f}% fully collapsed)')

---
## E3 — Mixed Reported + Discovered Urn: Eq. 2 & 3 Verification

**Eq. 2 (paper)**: When both reported ($w_r$) and discovered ($w_d = 1-w_r$) incidents are used,  
the limiting fraction $x^*$ satisfies:
$$x^* = \nu - \sqrt{\nu^2 - \frac{w_r r_A}{w_d(d_B - d_A)}}$$
where $\nu = \frac{1}{2} + \frac{R}{2\Delta_d}$, $R = w_r(r_A + r_B)$, $\Delta_d = w_d(\lambda_A - \lambda_B)$.

**Eq. 3**: Rewritten as $x^* = \frac{1+\kappa}{2} - \sqrt{\left(\frac{1+\kappa}{2}\right)^2 - \lambda^* \kappa}$  
where $\kappa = R/\Delta_d$. As $\kappa \to \infty$ (reported dominates), $x^* \to \lambda^*$ (correct answer).

**FTTL**: "Reported incidents" = pre-ML era claims (human handlers, not model-driven).  
We vary $w_r$ from 0 to 1 and verify theoretical $x^*$ matches simulation.

In [ ]:
def x_star_theory(lam_weak, lam_strong, r_weak, r_strong, w_r):
    """
    Theoretical fixed point x* from Eq. 2 of p12.
    x* = limiting fraction of WEAK (A = moderate) balls.
    Convention: lam_weak < lam_strong, so Δ_d = w_d*(λ_strong - λ_weak) > 0.

    Eq. 2:  x* = ν - √(ν² - w_r·r_weak / Δ_d)
    where   ν  = 1/2 + R/(2Δ_d),  R = w_r*(r_weak + r_strong)

    Boundary cases:
      w_r=0  → discovered only → x*=0  (Lemma 4: all scraps go to severe)
      w_r=1  → reported only   → x*=r_weak/(r_weak+r_strong) = ideal
    """
    w_d = 1.0 - w_r
    if w_d < 1e-9:
        return r_weak / (r_weak + r_strong)
    delta_d = w_d * (lam_strong - lam_weak)   # > 0 since lam_strong > lam_weak
    R = w_r * (r_weak + r_strong)
    nu = 0.5 + R / (2.0 * delta_d)
    discriminant = nu**2 - w_r * r_weak / delta_d
    if discriminant < 0:
        return nu
    return nu - np.sqrt(discriminant)

def simulate_mixed_urn(n_A, n_B, lam_A, lam_B, r_A, r_B, w_r, T, n_runs, rng):
    """
    Mixed urn: discovered (weight w_d) + reported (weight w_r).
    Reported incidents contribute r_A/r_B balls per step regardless of which segment was visited.
    Returns final x = fraction of A (moderate/weak) balls.
    """
    w_d = 1.0 - w_r
    finals = np.empty(n_runs)
    for run in range(n_runs):
        nA, nB = float(n_A), float(n_B)
        for _ in range(T):
            if rng.random() < nA / (nA + nB):
                if rng.random() < lam_A:
                    nA += w_d
            else:
                if rng.random() < lam_B:
                    nB += w_d
            # Reported: add proportional to true rate every step
            nA += w_r * r_A
            nB += w_r * r_B
        finals[run] = nA / (nA + nB)
    return finals

# Per Assumption 3.3: reported incidents track true total loss rate
r_weak, r_strong = lam_weak, lam_strong

w_r_values = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
T_E3, N_RUNS_E3 = 2000, 400

theory_vals = [x_star_theory(lam_weak, lam_strong, r_weak, r_strong, w) for w in w_r_values]
sim_means, sim_stds = [], []

for w_r in w_r_values:
    f = simulate_mixed_urn(N_WEAK_SMALL, N_STRONG_SMALL,
                           lam_weak, lam_strong, r_weak, r_strong,
                           w_r, T_E3, N_RUNS_E3, rng)
    sim_means.append(f.mean())
    sim_stds.append(f.std())

# κ = R/Δ_d for Eq. 3
kappa_vals, x_eq3_vals = [], []
for w_r in w_r_values:
    w_d = 1 - w_r
    if w_d < 1e-9:
        kappa_vals.append(np.nan); x_eq3_vals.append(x_ideal)
        continue
    delta_d = w_d * (lam_strong - lam_weak)
    R       = w_r * (r_weak + r_strong)
    kappa   = R / delta_d
    kappa_vals.append(kappa)
    # Eq. 3 rewrite: x* = (1+κ)/2 - √((1+κ/2)² - λ*·κ)
    x_eq3 = (1+kappa)/2 - np.sqrt(((1+kappa)/2)**2 - x_ideal*kappa)
    x_eq3_vals.append(x_eq3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(w_r_values, theory_vals, 'r-o', lw=2, ms=7, label='Eq. 2 theoretical x*')
ax.errorbar(w_r_values, sim_means, yerr=sim_stds,
            fmt='b--s', lw=1.5, ms=6, capsize=4, label='simulated (mean ± std)')
ax.axhline(x_ideal, color='green', ls=':', lw=1.5, label=f'ideal x*={x_ideal:.3f}')
ax.axhline(0.0, color='black', ls=':', lw=1, alpha=0.5, label='Lemma 4 limit (w_r=0)')
ax.set_xlabel('w_r (weight of reported / unbiased incidents)')
ax.set_ylabel('x* = limiting fraction of moderate scraps')
ax.set_title('E3a: Eq. 2 — more reported incidents → x* approaches ideal\n'
             'w_r=0: all scraps to severe; w_r=1: correct allocation')
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 0.5)

ax2 = axes[1]
valid = [(k, x) for k, x in zip(kappa_vals, x_eq3_vals) if not np.isnan(k) and k > 0]
kv, xv = zip(*valid)
ax2.semilogx(kv, xv, 'r-o', lw=2, ms=7, label='Eq. 3: x*(κ)')
ax2.axhline(x_ideal, color='green', ls='--', lw=1.5, label=f'ideal x*={x_ideal:.3f}')
ax2.axhline(0.0, color='black', ls=':', lw=1, alpha=0.5)
ax2.set_xlabel('κ = R/Δ_d  (log scale)\n← discovered dominates    reported dominates →')
ax2.set_ylabel('x* = limiting moderate fraction')
ax2.set_title('E3b: Eq. 3 — κ controls bias severity\n'
              'κ→0: x*→0 (collapse); κ→∞: x*→ideal (unbiased)')
ax2.legend(fontsize=8)
ax2.set_ylim(-0.05, 0.4)

plt.tight_layout()
plt.savefig('p12_e3_mixed_urn.png', dpi=150, bbox_inches='tight')
plt.show()

print('Eq. 2 verification (theory vs simulation):')
print(f'{"w_r":>4}  {"theory":>8}  {"sim":>8}  {"std":>8}')
for w_r, th, sm, st in zip(w_r_values, theory_vals, sim_means, sim_stds):
    print(f'{w_r:>4.1f}  {th:>8.4f}  {sm:>8.4f}  {st:>8.4f}')

---
## E4 — FTTL Empirical Urn Trajectory (Fig. 1 Analog)

We track the **empirical urn** using real data: at each quarter, the urn proportion  
= fraction of scrap decisions (model_v1_decision=1) going to each segment.

**What to look for:**
- Urn proportion for **severe** increasing over time → Lemma 4 prediction confirmed
- Compare `model_v1_decision` urn (no fix) vs counterfactual urn weighted by 1/scrap_rate (IPS fix)
- The urn should drift away from the ideal λ* = 0.698 toward 1.0

In [ ]:
df['quarter'] = df['claim_date'].dt.to_period('Q')
quarters  = sorted(df['quarter'].unique())
qlabels   = [str(q) for q in quarters]
t         = np.arange(len(quarters))
xtick_idx = list(range(0, len(quarters), 4))

# x = fraction of MODERATE in cumulative scraps (should decrease toward 0 per Lemma 4)
cum_mod, cum_sev = 0.0, 0.0
cum_true_mod, cum_true_sev = 0.0, 0.0

x_empirical = []   # biased (model_v1_decision)
x_oracle    = []   # unbiased (pre_ml_label)

for q in quarters:
    sub = df[df['quarter'] == q]
    cum_mod += sub[sub['damage_severity']=='moderate']['model_v1_decision'].sum()
    cum_sev += sub[sub['damage_severity']=='severe']['model_v1_decision'].sum()
    cum_true_mod += sub[sub['damage_severity']=='moderate']['pre_ml_label'].sum()
    cum_true_sev += sub[sub['damage_severity']=='severe']['pre_ml_label'].sum()

    denom = cum_mod + cum_sev
    x_empirical.append(cum_mod / denom if denom > 0 else np.nan)
    denom_true = cum_true_mod + cum_true_sev
    x_oracle.append(cum_true_mod / denom_true if denom_true > 0 else np.nan)

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

ax = axes[0]
ax.plot(t, x_empirical, 'r-o', ms=4, lw=1.5, label='empirical x (model_v1_decision, biased)')
ax.plot(t, x_oracle,    'g--s', ms=4, lw=1.5, label='oracle x (pre_ml_label, unbiased)')
ax.axhline(x_ideal, color='blue', ls=':', lw=1.5, label=f'ideal x*={x_ideal:.3f}')
ax.axhline(0.0, color='black', ls='--', lw=1, alpha=0.4, label='Lemma 4 limit (x→0)')
ax.axvline(19, color='gray', ls='--', lw=1, alpha=0.7, label='v1 deployment (2021-Q2)')
ax.set_ylabel('x = moderate fraction of cumulative scraps')
ax.set_title('E4a: Empirical Urn Trajectory\n'
             'Lemma 4 predicts x→0 (all scraps to severe) — does the data show this drift?')
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 0.8)

# Stacked bar of cumulative balls (moderate vs severe vs minor)
cum_min = 0.0
stk_mod, stk_sev, stk_min = [], [], []
for q in quarters:
    sub = df[df['quarter'] == q]
    cum_min += sub[sub['damage_severity']=='minor']['model_v1_decision'].sum()
    stk_mod.append(cum_mod := stk_mod[-1] + sub[sub['damage_severity']=='moderate']['model_v1_decision'].sum()
                   if stk_mod else sub[sub['damage_severity']=='moderate']['model_v1_decision'].sum())
    stk_sev.append(cum_sev := stk_sev[-1] + sub[sub['damage_severity']=='severe']['model_v1_decision'].sum()
                   if stk_sev else sub[sub['damage_severity']=='severe']['model_v1_decision'].sum())
    stk_min.append(cum_min)

ax2 = axes[1]
ax2.stackplot(t, [stk_sev, stk_mod, stk_min],
              labels=['severe', 'moderate', 'minor'],
              colors=['#F44336', '#FF9800', '#4CAF50'], alpha=0.75)
ax2.axvline(19, color='gray', ls='--', lw=1, alpha=0.7)
ax2.set_xlabel('quarter')
ax2.set_ylabel('cumulative scrap decisions')
ax2.set_title('E4b: Urn ball counts per segment — severe dominates (Lemma 4 direction)')
ax2.legend(fontsize=8, loc='upper left')
ax2.set_xticks(xtick_idx)
ax2.set_xticklabels([qlabels[i] for i in xtick_idx], rotation=45, ha='right')

plt.tight_layout()
plt.savefig('p12_e4_empirical_urn.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'x_empirical: start={x_empirical[0]:.4f} → end={x_empirical[-1]:.4f}  '
      f'({"↓ toward 0 — Lemma 4 direction" if x_empirical[-1] < x_empirical[0] else "↑ stable/reversed"})')
print(f'x_oracle:    start={x_oracle[0]:.4f} → end={x_oracle[-1]:.4f}  (unbiased baseline)')
print(f'Ideal x*: {x_ideal:.4f}')

---
## E5 — Rejection Sampling Fix (Section 3.5)

**The fix (Section 3.5)**: Instead of always adding a ball when a total loss is discovered,  
first sample another ball. Only add if the **colours differ** (rejection sampling).  
This down-weights regions visited more often, making the update proportional to $\lambda_i$ alone.

**Importance-sampling analog**: weight = $1 / \Pr(\text{scrap})$ per segment = IPS.  
This is precisely the **Thompson-Horvitz estimator**.

**FTTL implementation**: IPS weight for claim $i$ = $1 / \hat{p}_{\text{scrap}}(\text{segment}_i)$,  
where $\hat{p}_{\text{scrap}}$ is the scrap rate observed per segment over time.

We compare: uncorrected urn (Lemma 4 collapse) vs rejection-sampling-corrected urn (converges to λ*).

In [ ]:
def simulate_rejection_fix(n_A, n_B, lam_A, lam_B, T, n_runs, rng):
    """
    Rejection sampling fix (Section 3.5).
    After discovering a total loss in region i, sample a second ball.
    Add a new ball for region i ONLY if the second draw is from the OTHER region.
    Net effect: P(add A ball) ∝ λ_A · (n_B/(n_A+n_B)), independent of urn composition after many steps.
    This makes the update proportional to λ_i alone — converges to x* = λ_A/(λ_A+λ_B) = x_ideal.
    """
    finals = np.empty(n_runs)
    for run in range(n_runs):
        nA, nB = float(n_A), float(n_B)
        for _ in range(T):
            pA = nA / (nA + nB)
            if rng.random() < pA:           # visit A (moderate)
                if rng.random() < lam_A:    # total loss found
                    if rng.random() >= pA:  # second draw is B → accept
                        nA += 1
            else:                           # visit B (severe)
                if rng.random() < lam_B:
                    if rng.random() < pA:   # second draw is A → accept
                        nB += 1
        finals[run] = nA / (nA + nB)
    return finals

T_E5, N_RUNS_E5 = 8000, 600

finals_no_fix = simulate_discovered_only(N_WEAK_SMALL, N_STRONG_SMALL,
                                          lam_weak, lam_strong, T_E5, N_RUNS_E5, rng)
finals_fix    = simulate_rejection_fix(N_WEAK_SMALL, N_STRONG_SMALL,
                                        lam_weak, lam_strong, T_E5, N_RUNS_E5, rng)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(finals_no_fix, bins=40, density=True, color='#F44336', alpha=0.7, label='no fix (Lemma 4)')
ax.hist(finals_fix,    bins=40, density=True, color='#2196F3', alpha=0.7, label='rejection sampling fix')
ax.axvline(x_ideal, color='green', ls='--', lw=2, label=f'ideal x*={x_ideal:.3f}')
ax.axvline(0.0, color='black', ls=':', lw=1.5, label='Lemma 4 collapse')
ax.set_xlabel('x = fraction of moderate (weak) balls')
ax.set_ylabel('density')
ax.set_title(f'E5a: Simulation (T={T_E5}, {N_RUNS_E5} runs)\n'
             f'Fix centres distribution near x*={x_ideal:.3f} vs collapse to 0')
ax.legend(fontsize=8)

# IPS-weighted empirical urn (Thompson-Horvitz analog)
# Weight each scrap decision by 1/segment_scrap_rate → down-weights over-scrapped segments
scrap_rate_seg = df.groupby('damage_severity')['model_v1_decision'].mean()
df['ips_w'] = df['damage_severity'].map(
    lambda s: 1.0 / scrap_rate_seg[s] if scrap_rate_seg[s] > 1e-6 else 0.0
)

x_ips = []
ips_cum_mod, ips_cum_sev = 0.0, 0.0
for q in quarters:
    sub = df[df['quarter'] == q]
    mod_sub = sub[sub['damage_severity']=='moderate']
    sev_sub = sub[sub['damage_severity']=='severe']
    ips_cum_mod += (mod_sub['model_v1_decision'] * mod_sub['ips_w']).sum()
    ips_cum_sev += (sev_sub['model_v1_decision'] * sev_sub['ips_w']).sum()
    denom = ips_cum_mod + ips_cum_sev
    x_ips.append(ips_cum_mod / denom if denom > 0 else np.nan)

ax2 = axes[1]
ax2.plot(t, x_empirical, 'r-',  lw=1.5, label='unweighted urn (biased)')
ax2.plot(t, x_ips,       'b--', lw=1.5, label='IPS-weighted urn (Section 3.5 analog)')
ax2.plot(t, x_oracle,    'g:',  lw=1.5, label='oracle (pre_ml_label, unbiased)')
ax2.axhline(x_ideal, color='green', ls='-', lw=1, alpha=0.4, label=f'ideal x*={x_ideal:.3f}')
ax2.axhline(0.0, color='black', ls=':', lw=1, alpha=0.3)
ax2.set_xlabel('quarter')
ax2.set_ylabel('x = moderate fraction of cumulative scraps')
ax2.set_title('E5b: IPS-weighted empirical urn vs unweighted\n'
              '(Thompson-Horvitz estimator, Section 3.5)')
ax2.set_xticks(xtick_idx)
ax2.set_xticklabels([qlabels[i] for i in xtick_idx], rotation=45, ha='right', fontsize=7)
ax2.legend(fontsize=8)
ax2.set_ylim(-0.05, 0.8)

plt.tight_layout()
plt.savefig('p12_e5_rejection_fix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'No-fix x mean: {finals_no_fix.mean():.4f}  (ideal: {x_ideal:.4f},  error: {abs(finals_no_fix.mean()-x_ideal):.4f})')
print(f'Fix    x mean: {finals_fix.mean():.4f}  (ideal: {x_ideal:.4f},  error: {abs(finals_fix.mean()-x_ideal):.4f})')
print(f'Fix error reduction: {(1 - abs(finals_fix.mean()-x_ideal)/abs(finals_no_fix.mean()-x_ideal))*100:.1f}%')

---
## Summary

| Experiment | p12 claim | Verified? |
|---|---|---|
| E1 | Lemma 2: standard urn → Beta(n_r, n_b); initial bias locked in | KS test vs Beta |
| E2 | Lemma 4: discovered-only urn → 100% to winner even with tiny λ gap | % runs > 0.95 |
| E3 | Eq. 2 & 3: more reported incidents (larger κ) pulls x* toward λ* | theory vs sim |
| E4 | Empirical urn drifts away from ideal λ* after model deployment | trajectory plot |
| E5 | Rejection sampling (IPS) fix restores convergence to λ* | mean error reduction |

**Key FTTL finding from Lemma 4**: Even if `damage_severity=moderate` and `severe` had only a 0.01 difference in total loss rate,
the discovered-only feedback loop would still concentrate **all** scrap decisions on the marginally higher segment.
The IPS fix (E5) — equivalent to weighting training examples by $1/\hat{p}_{\text{scrap}}$ — is the
mathematical justification for **Build 03 unbiased evaluation** and **Build 06 IPW reweighting**.